# 图片

In [3]:
import os
import io
import uuid
from PIL import Image
from flask import Flask, request, jsonify

app = Flask(__name__)

# 设置保存图片的目录
UPLOAD_FOLDER = os.path.join('D:', 'Downloads')  # 这里使用了 os.path.join 来确保兼容性
if not os.path.exists(UPLOAD_FOLDER):
    os.makedirs(UPLOAD_FOLDER)

# 限制上传的文件类型
allowed_extensions = {'png', 'jpg', 'jpeg', 'gif'}

def allowed_file(filename):
    """检查文件扩展名是否在允许的范围内"""
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in allowed_extensions

@app.route('/predict_facemoe', methods=['POST'])
def predict_facemoe():
    # 确保请求中包含图像文件
    if 'image' not in request.files:
        return jsonify({'error': 'No image provided'}), 400

    file = request.files['image']

    # 检查文件类型
    if not allowed_file(file.filename):
        return jsonify({'error': 'Invalid file type'}), 400

    try:
        # 输出文件信息，方便调试
        print("接收到文件：", file.filename)
        print("文件类型：", file.content_type)

        # 打开图片文件
        image = Image.open(io.BytesIO(file.read()))
        print("图片处理成功，开始保存...")

        # 生成新的唯一文件名，避免覆盖
        file_name, file_extension = os.path.splitext(file.filename)
        new_file_name = str(uuid.uuid4()) + file_extension
        file_path = os.path.join(UPLOAD_FOLDER, new_file_name)

        # 保存图片到本地
        image.save(file_path)
        print(f"图片已成功保存到：{file_path}")

        # 返回成功信息和文件路径
        return jsonify({'message': 'Image saved successfully', 'file_path': file_path})

    except Exception as e:
        # 捕获异常并打印
        print(f"出现错误: {e}")
        return jsonify({'error': f'Error processing image: {str(e)}'}), 500

if __name__ == '__main__':
    # 设置 Flask 服务器开启调试模式
    app.run(debug=True)


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with stat


SystemExit: 1

C:\Users\27641\AppData\Roaming\Python\Python38\site-packages\IPython\core\interactiveshell.py:3516: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# 语音

In [ ]:
import os
import uuid
from flask import Flask, request, jsonify

app = Flask(__name__)

# 设置保存音频文件的目录
UPLOAD_FOLDER = os.path.join('D:', 'Downloads')
if not os.path.exists(UPLOAD_FOLDER):
    os.makedirs(UPLOAD_FOLDER)

# 限制上传的文件类型
allowed_extensions = {'mp3', 'wav', 'ogg'}

def allowed_file(filename):
    """检查文件扩展名是否在允许的范围内"""
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in allowed_extensions



@app.route('/classify_audio2emo', methods=['POST'])
def classify_audio2emo():
    # 确保请求中包含音频文件
    if 'audio' not in request.files:
        return jsonify({'error': 'No audio file provided'}), 400

    file = request.files['audio']


    # 检查文件类型
    if not allowed_file(file.filename):
        return jsonify({'error': 'Invalid file type'}), 400

    try:
        # 输出文件信息，方便调试
        print("接收到文件：", file.filename)
        print("文件类型：", file.content_type)

        # 生成新的唯一文件名，避免覆盖
        file_name, file_extension = os.path.splitext(file.filename)
        new_file_name = str(uuid.uuid4()) + file_extension
        file_path = os.path.join(UPLOAD_FOLDER, new_file_name)

        # 保存音频文件到本地
        with open(file_path, 'wb') as f:
            f.write(file.read())
        print(f"音频文件已成功保存到：{file_path}")

        # 返回成功信息和文件路径
        return jsonify({'message': 'Audio saved successfully', 'file_path': file_path})

    except Exception as e:
        # 捕获异常并打印
        print(f"出现错误: {e}")
        return jsonify({'error': f'Error processing audio: {str(e)}'}), 500

if __name__ == '__main__':
    # 设置 Flask 服务器开启调试模式
    app.run(debug=True)


# 多文件

In [ ]:
import os
import uuid
import logging
from flask import Flask, request, jsonify

app = Flask(__name__)

# 设置保存音频文件的目录
app.config['UPLOAD_FOLDER'] = 'D:/Downloads'
if not os.path.exists(app.config['UPLOAD_FOLDER']):
    os.makedirs(app.config['UPLOAD_FOLDER'])

# 限制上传的文件类型
ALLOWED_EXTENSIONS = {'mp3', 'wav', 'ogg'}

# 设置日志格式
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def allowed_file(filename):
    """检查文件扩展名是否在允许的范围内"""
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in ALLOWED_EXTENSIONS


@app.route('/verify_audioid', methods=['POST'])
def verify_audioid():
    # 检查请求是否包含两个音频文件
    if 'speaker1' not in request.files or 'speaker2' not in request.files:
        return jsonify({'error': 'Both audio files must be provided'}), 400

    files = {
        'speaker1': request.files['speaker1']
    }
    
    saved_files = {}
    try:
        for key, file in files.items():
            # 检查文件类型
            if not allowed_file(file.filename):
                return jsonify({'error': f'Invalid file type for {key}'}), 400

            # 生成新的唯一文件名，避免覆盖
            file_extension = os.path.splitext(file.filename)[1]
            new_file_name = f"{uuid.uuid4()}{file_extension}"
            file_path = os.path.join(app.config['UPLOAD_FOLDER'], new_file_name)

            # 保存音频文件到本地
            file.save(file_path)
            logging.info(f"音频文件 {key} 已成功保存到：{file_path}")

            # 添加文件信息到结果
            saved_files[key] = new_file_name

        # 返回成功信息和保存的文件名
        return jsonify({'message': 'Both audio files saved successfully', 'saved_files': saved_files})

    except Exception as e:
        logging.error(f"音频文件保存失败: {e}")
        return jsonify({'error': f'Error processing audio files: {str(e)}'}), 500

if __name__ == '__main__':
    app.run(debug=True, host='0.0.0.0', port=5000)


# voice2User

In [ ]:
import requests

# 发送 POST 请求
url = 'http://bt4g6s.natappfree.cc/verify_audioid'
files = {
    'speaker1': open('speaker.wav', 'rb'),
}
response = requests.post(url, files=files)

# 打印预测结果
print(response.json())

[{'filename': 'czx.wav', 'score': 0.99681}]


In [4]:
import os
import uuid
from flask import Flask, request, jsonify
import soundfile as sf

app = Flask(__name__)

# 设置保存音频文件的目录
UPLOAD_FOLDER = os.path.join('D:', 'Downloads')
if not os.path.exists(UPLOAD_FOLDER):
    os.makedirs(UPLOAD_FOLDER)

# 限制上传的文件类型
allowed_extensions = {'mp3', 'wav', 'ogg'}

def allowed_file(filename):
    """检查文件扩展名是否在允许的范围内"""
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in allowed_extensions


@app.route('/verify_audioid', methods=['POST'])
def verify_audioid():
    if 'speaker1' not in request.files:
        return jsonify({'error': 'One audio file must be provided'}), 400
    
    # 读取 speaker1 的音频数据
    file = request.files['speaker1']

    speaker1_audio = file.read()

    speaker1_wav, sample_rate = sf.read(io.BytesIO(speaker1_audio))


    print(sample_rate)
    print(speaker1_wav)
    
    # 检查文件类型
    if not allowed_file(file.filename):
        return jsonify({'error': 'Invalid file type'}), 400

    try:
        # 输出文件信息，方便调试
        print("接收到文件：", file.filename)
        print("文件类型：", file.content_type)

        # 生成新的唯一文件名，避免覆盖
        file_name, file_extension = os.path.splitext(file.filename)
        new_file_name = str(uuid.uuid4()) + file_extension
        file_path = os.path.join(UPLOAD_FOLDER, new_file_name)

        # 保存音频文件到本地
        with open(file_path, 'wb') as f:
            f.write(file.read())
        print(f"音频文件已成功保存到：{file_path}")

        # 返回成功信息和文件路径
        return jsonify({'message': 'Audio saved successfully', 'file_path': file_path})

    except Exception as e:
        # 捕获异常并打印
        print(f"出现错误: {e}")
        return jsonify({'error': f'Error processing audio: {str(e)}'}), 500

if __name__ == '__main__':
    # 设置 Flask 服务器开启调试模式
    app.run(debug=True)


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with stat


SystemExit: 1

C:\Users\27641\AppData\Roaming\Python\Python38\site-packages\IPython\core\interactiveshell.py:3516: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
@app.route('/verify_audioid', methods=['POST'])
def verify_audioid():
    if 'speaker1' not in request.files:
        return jsonify({'error': 'One audio file must be provided'}), 400
    
    # 读取 speaker1 的音频数据
    speaker1_audio = request.files['speaker1'].read()

    # 使用 soundfile 读取音频数据及其采样率
    speaker1_wav, sample_rate = sf.read(io.BytesIO(speaker1_audio))
        # 如果是立体声，选择第一个通道并将其转换为单声道